<a href="https://colab.research.google.com/github/Adidtiasmara/pembelajaran-mesin/blob/main/JS05/JS05-Tugas%2002.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas Lab 2: Klasifikasi SMS Spam dengan Multinomial Naive Bayes




## 1. Import Library

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

## 2. Load Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Encoding latin-1 diperlukan karena data bukan UTF-8
df = pd.read_csv('/content/drive/MyDrive/MachineLearning/spam.csv', encoding='latin-1')

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


Terdapat tiga kolom kosong yang tidak dipakai. Kolom tersebut dibuang, lalu nama kolom `v1` (label) dan `v2` (teks SMS) diubah agar mudah dibaca.

In [5]:
# Buang 3 kolom terakhir
df = df.drop(df.iloc[:, 2:], axis=1)

# Ubah nama kolom
df = df.rename(columns={'v1': 'Labels', 'v2': 'SMS'})

df.head()

,Labels,SMS
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
# Jumlah data per kelas
print(df['Labels'].value_counts())
print('\n')

# Kelengkapan data
print(df.info())
print('\n')

# Statistik deskriptif
print(df.describe())

Labels
ham     4825
spam     747
Name: count, dtype: int64


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Labels  5572 non-null   object
 1   SMS     5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


       Labels                     SMS
count    5572                    5572
unique      2                    5169
top       ham  Sorry, I'll call later
freq     4825                      30


## 3. Preprocessing

In [7]:
# Encoding label: spam = 1, ham = 0
df['Labels'] = df['Labels'].map({'spam': 1, 'ham': 0})

df.head()

,Labels,SMS
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
X = df['SMS'].astype(str).to_numpy(dtype=object)
y = df['Labels'].to_numpy(dtype=int)

print('Jumlah data:', len(X))
print('Jumlah label:', len(y))

Jumlah data: 5572
Jumlah label: 5572


Data dibagi lebih dulu, baru dilakukan transformasi. Data latih memakai `fit_transform`, data uji hanya `transform` agar model tidak mengetahui data uji (mencegah kebocoran informasi).

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=50
)

print('Data latih:', len(X_train))
print('Data uji:', len(X_test))

Data latih: 4457
Data uji: 1115


## 4. Tugas 1: CountVectorizer dengan stop_words

`CountVectorizer` menghitung jumlah kemunculan setiap kata. Parameter `stop_words='english'` membuang kata umum yang tidak bermakna seperti *the*, *is*, dan *and*.

In [10]:
bow = CountVectorizer(stop_words='english')

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

print('Jumlah fitur:', len(bow.get_feature_names_out()))
print('Dimensi data latih:', X_train_bow.shape)

Jumlah fitur: 7466
Dimensi data latih: (4457, 7466)


In [11]:
mnb_bow = MultinomialNB()
mnb_bow.fit(X_train_bow, y_train)

acc_train_bow = accuracy_score(y_train, mnb_bow.predict(X_train_bow))
acc_test_bow = accuracy_score(y_test, mnb_bow.predict(X_test_bow))

print('Akurasi data latih:', acc_train_bow)
print('Akurasi data uji:', acc_test_bow)
print('\nLaporan Klasifikasi:')
print(classification_report(y_test, mnb_bow.predict(X_test_bow), target_names=['ham', 'spam']))

Akurasi data latih: 0.9946152120260264
Akurasi data uji: 0.9829596412556054

Laporan Klasifikasi:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       954
        spam       0.98      0.90      0.94       161

    accuracy                           0.98      1115
   macro avg       0.98      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



## 5. Tugas 2: TF-IDF dengan stop_words

`TfidfVectorizer` memberi bobot pada setiap kata. Kata yang sering muncul di banyak SMS diberi bobot lebih kecil, sedangkan kata yang jarang tetapi khas diberi bobot lebih besar.

In [12]:
tfidf = TfidfVectorizer(stop_words='english')

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print('Jumlah fitur:', len(tfidf.get_feature_names_out()))
print('Dimensi data latih:', X_train_tfidf.shape)

Jumlah fitur: 7466
Dimensi data latih: (4457, 7466)


In [13]:
mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)

acc_train_tfidf = accuracy_score(y_train, mnb_tfidf.predict(X_train_tfidf))
acc_test_tfidf = accuracy_score(y_test, mnb_tfidf.predict(X_test_tfidf))

print('Akurasi data latih:', acc_train_tfidf)
print('Akurasi data uji:', acc_test_tfidf)
print('\nLaporan Klasifikasi:')
print(classification_report(y_test, mnb_tfidf.predict(X_test_tfidf), target_names=['ham', 'spam']))

Akurasi data latih: 0.9842943684092439
Akurasi data uji: 0.9605381165919282

Laporan Klasifikasi:
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       954
        spam       1.00      0.73      0.84       161

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



## 6. Perbandingan Hasil

In [14]:
perbandingan = pd.DataFrame({
    'Fitur': ['CountVectorizer', 'TF-IDF'],
    'Akurasi Latih': [acc_train_bow, acc_train_tfidf],
    'Akurasi Uji': [acc_test_bow, acc_test_tfidf]
})

perbandingan

,Fitur,Akurasi Latih,Akurasi Uji
0,CountVectorizer,0.994615,0.982960
1,TF-IDF,0.984294,0.960538


## 7. Kesimpulan

Berdasarkan hasil pengujian:

- **CountVectorizer dengan stop_words** menghasilkan akurasi data uji sekitar 0,983.
- **TF-IDF dengan stop_words** menghasilkan akurasi data uji sekitar 0,961.

Pada kasus data `spam.csv` ini, **CountVectorizer** lebih baik daripada TF-IDF. Penyebabnya adalah SMS spam dikenali dari kata-kata tertentu yang muncul berulang, misalnya *free*, *win*, *txt*, dan *claim*. `CountVectorizer` menghitung kemunculan kata tersebut secara langsung, sedangkan TF-IDF menurunkan bobot kata yang sering muncul sehingga sinyal tersebut menjadi lebih lemah.

Jadi, fitur terbaik untuk data `spam.csv` pada percobaan ini adalah **CountVectorizer dengan stop_words**.